# **📚 PDF RAG Chatbot**

### A Retrieval-Augmented Generation (RAG) system that allows users
### to ask questions from PDF documents using LangChain, HuggingFace,
### Chroma and Groq.

## **1. Project Introduction**

### This project is a Retrieval-Augmented Generation (RAG) chatbot that allows users to ask questions about a PDF document using natural language. The system loads the Python Built-In Functions PDF, extracts its text, divides the content into smaller chunks, and converts those chunks into vector embeddings using a Hugging Face embedding model. These embeddings are stored in ChromaDB, which allows the system to retrieve the most relevant information for a user's question. Finally, a Groq-hosted LLM uses the retrieved content to generate a relevant answer. The chatbot is designed to answer based on the provided document and avoid generating answers when the required information is not available.

## **2. Install Libraries**


In [ ]:
#!pip install --upgrade langchain
#!pip install langchain_community
#!pip install langchain_text_splitter
#!pip install langchain_chroma
#!pip install langchain_groq
#!pip install langchain


In [ ]:
#!pip install unstructured

In [ ]:
#!pip install -qU langchain-community beautifulsoup4

In [ ]:
#!pip install -qU pypdf

In [29]:
#!pip install -q langchain-community langchain-chroma langchain-groq pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137

In [24]:
!pip list | grep -E "langchain|pypdf"

langchain                             1.3.18
langchain-core                        1.6.1
langchain-protocol                    0.0.19


In [25]:
!pip list | grep langchain

langchain                             1.3.18
langchain-core                        1.6.1
langchain-protocol                    0.0.19


In [26]:
!pip list | grep -E "langchain|chroma|groq|sentence-transformers|pypdf|beautifulsoup4"

beautifulsoup4                        4.13.5
langchain                             1.3.18
langchain-core                        1.6.1
langchain-protocol                    0.0.19
sentence-transformers                 5.7.0


In [28]:
import sys

print("Python version:", sys.version)
print("Python location:", sys.executable)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Python location: /usr/bin/python3


In [22]:
import sys
print(sys.executable)

/usr/bin/python3


In [30]:
from langchain_community.document_loaders import PyPDFLoader

print("✅ PyPDFLoader is ready!")

/tmp/ipykernel_784/2713864483.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ PyPDFLoader is ready!


## **3. Import Libraries**

In [31]:
import os
#from langchain.document_loaders import UnstructuredFileLoader #old import not working
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA


## **4. API Key Configuration**

In [32]:
import os

os.environ["GROQ_API_KEY"] = "USE_YOUR_API_KEY"

## **5. Load PDF / URL**

In [33]:
import requests
url = "https://dspmuranchi.ac.in/pdf/Blog/Python%20Built-In%20Functions.pdf"

## **6. Information Extraction**

In [34]:
from langchain_community.document_loaders import PyPDFLoader

url = "https://dspmuranchi.ac.in/pdf/Blog/Python%20Built-In%20Functions.pdf"

loader = PyPDFLoader(url)

documents = loader.load()

print("Number of pages:", len(documents))
print(documents[0].page_content[:1000])

Number of pages: 15
Python Built-In Functions 
 
Gaurav Kr. suman       MIT5


In [35]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 15


## **7. Text Splitting**

In [36]:
print(documents[0].page_content)

Python Built-In Functions 
 
Gaurav Kr. suman       MIT5


In [37]:
for i, doc in enumerate(documents):
    print(f"\n--- Page {i+1} ---")
    print(doc.page_content[:500])


--- Page 1 ---
Python Built-In Functions 
 
Gaurav Kr. suman       MIT5

--- Page 2 ---
1 | P a g e  
 
1. abs() 
The abs() is one of the most popular Python built -in functions, 
which returns the absolute value of a number. A negative value’s 
absolute is that value is positive.  
>>>  ab s(-7) 
7 
>>>  ab s(7) 
7 
>>>  ab s(0) 
2. all() 
The  all() function takes a container as an argument. This Built in 
Functions returns True if all values in a python iterable have a 
Boolean value of True. An empty value has a Boolean value of 
False.  
>>>  al l({'* ','',''}) 
False  
>>>  a

--- Page 3 ---
2 | P a g e  
 
>>>  ascii ('ș' ) 
“‘\\u0219′”  
Since this was a non -ASCII character in python, the interpreter 
added a backslash ( \) and escaped it using another backslash.  
>>>  ascii ('ușor' ) 
“‘u \\u0219or'”  
Let’s apply it to a list.  
>>>  asci i(['s ','ș ']) 
“[‘s’, ‘ \\u0219’]”  
5. bin() 
bin()  converts an integer to a binary string. We have seen this and 
other functions in

In [38]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

## **8. Chunking**

In [39]:
print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

Total chunks: 15

--- Chunk 1 ---
Python Built-In Functions 
 
Gaurav Kr. suman       MIT5

--- Chunk 2 ---
1 | P a g e  
 
1. abs() 
The abs() is one of the most popular Python built -in functions, 
which returns the absolute value of a number. A negative value’s 
absolute is that value is positive.  
>>>  ab s(-7) 
7 
>>>  ab s(7) 
7 
>>>  ab s(0) 
2. all() 
The  all() function takes a container as an argument. This Built in 
Functions returns True if all values in a python iterable have a 
Boolean value of True. An empty value has a Boolean value of 
False.  
>>>  al l({'* ','',''}) 
False  
>>>  al l([' ',' ',' ']) 
True  
3. any() 
Like all(), it takes one argument and returns True if, even one 
value in the iterable has a Boolean value of True.  
>>>  an y((1,0,0)) 
True  
>>>  any (( 0,0,0))  
False  
4.  ascii() 
It is important  Python built -in functions,  returns a printable 
representation o f a  python  object  (like a string or a  Python  list ). 
Let’s take a Romanian ch

## **Inspect the Chunk**

In [40]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n========== CHUNK {i+1} ==========")
    print(chunk.page_content)


========== CHUNK 1 ==========
Python Built-In Functions 
 
Gaurav Kr. suman       MIT5

========== CHUNK 2 ==========
1 | P a g e  
 
1. abs() 
The abs() is one of the most popular Python built -in functions, 
which returns the absolute value of a number. A negative value’s 
absolute is that value is positive.  
>>>  ab s(-7) 
7 
>>>  ab s(7) 
7 
>>>  ab s(0) 
2. all() 
The  all() function takes a container as an argument. This Built in 
Functions returns True if all values in a python iterable have a 
Boolean value of True. An empty value has a Boolean value of 
False.  
>>>  al l({'* ','',''}) 
False  
>>>  al l([' ',' ',' ']) 
True  
3. any() 
Like all(), it takes one argument and returns True if, even one 
value in the iterable has a Boolean value of True.  
>>>  an y((1,0,0)) 
True  
>>>  any (( 0,0,0))  
False  
4.  ascii() 
It is important  Python built -in functions,  returns a printable 
representation o f a  python  object  (like a string or a  Python  list ). 
Let’s take a 

In [41]:
# Check the metadata
print(chunks[0].metadata)

{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2020-06-04T12:32:44+05:30', 'title': 'Python Built-In Functions', 'author': 'Gaurav Kr. suman', 'moddate': '2020-06-04T12:32:44+05:30', 'source': 'https://dspmuranchi.ac.in/pdf/Blog/Python%20Built-In%20Functions.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


## **9. Create Embeddings**

In [42]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_784/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## **10. Create Chroma Vector Database**

In [43]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="python_functions"
)

print("Documents stored in Chroma successfully!")

Documents stored in Chroma successfully!


In [44]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [45]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

## **11. Create Retriever**

In [46]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

## **Test Retrieval BEFORE using Groq**

In [47]:
question = "What are Python built-in functions?"

In [48]:
retrieved_docs = retriever.invoke(question)

In [49]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n========== RESULT {i+1} ==========")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:500])


========== RESULT 1 ==========
Page: 0
Python Built-In Functions 
 
Gaurav Kr. suman       MIT5

========== RESULT 2 ==========
Page: 1
1 | P a g e  
 
1. abs() 
The abs() is one of the most popular Python built -in functions, 
which returns the absolute value of a number. A negative value’s 
absolute is that value is positive.  
>>>  ab s(-7) 
7 
>>>  ab s(7) 
7 
>>>  ab s(0) 
2. all() 
The  all() function takes a container as an argument. This Built in 
Functions returns True if all values in a python iterable have a 
Boolean value of True. An empty value has a Boolean value of 
False.  
>>>  al l({'* ','',''}) 
False  
>>>  a

========== RESULT 3 ==========
Page: 5
5 | P a g e  
 
‘a’  
>>>  ch r(9) 
‘\t’ 
>>>  ch r(48) 
‘0’  
11. classmethod() 
classmethod() returns a class method for a given method.  
>>>  class  fruit : 
def  sayh i(sel f): 
prin t("Hi,  I'm  a fruit ")  
 
>>>  fruit.sayhi =classmetho d(fruit.sayh i) 
>>>  fruit .sayh i() 
Hi, I’m a fruit  
When we pass the met

## **12. Configure Groq LLM**

In [ ]:
response = llm.invoke("What are Python built-in functions?")
print(response.content)

## **13. Create RAG Chain**

In [50]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [51]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [52]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the question using ONLY the context provided below.

If the answer cannot be found in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
""")

## **Create context formatter**

In [53]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

## **Build the RAG pipeline**

In [54]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

## **14. Ask Questions**

In [55]:
question = "What are Python built-in functions?"

result = qa_chain.invoke({
    "query": question
})

print(result["result"])

**Python built‑in functions** are functions that are always available in the Python interpreter without needing to import any module. They are part of the core language and can be called directly from any Python program. These functions cover a wide range of common tasks such as mathematical operations, data conversion, object introspection, and more.

Below is a quick overview of some frequently used built‑in functions (the list is not exhaustive):

| Function | Purpose | Example |
|----------|---------|---------|
| `abs()` | Returns the absolute value of a number. | `abs(-7)` → `7` |
| `all()` | Returns `True` if all elements in an iterable are truthy. | `all([True, True])` → `True` |
| `any()` | Returns `True` if at least one element in an iterable is truthy. | `any([0, 1, 0])` → `True` |
| `ascii()` | Returns a printable representation of an object, escaping non‑ASCII characters. | `ascii('a')` → `'a'` |
| `classmethod()` | Transforms a method into a class method. | `@classmethod` 

In [56]:
question = "What is the purpose of the len() function?"

result = qa_chain.invoke({
    "query": question
})

print(result["result"])

The `len()` function is a built‑in Python function that returns the **length** of an object.  
In practice, it tells you how many items are contained in a sequence or collection:

- **Strings** – number of characters  
- **Lists, tuples, sets** – number of elements  
- **Dictionaries** – number of key‑value pairs (i.e., the number of keys)  

Example:

```python
len("hello")      # 5
len([1, 2, 3])    # 3
len({"a": 1, "b": 2})  # 2
```

So, `len()` is used whenever you need to know the size or count of elements in an iterable or container.


In [57]:
question = "Give me examples of Python built-in functions."

response = rag_chain.invoke(question)

print(response.content)

Here are some examples of Python built‑in functions that are mentioned in the provided context, along with short code snippets showing how they are used:

| Built‑in function | Purpose | Example |
|-------------------|---------|---------|
| **`abs()`** | Returns the absolute value of a number. | ```python<br>abs(-7)   # → 7<br>abs(7)    # → 7<br>abs(0)    # → 0``` |
| **`all()`** | Returns `True` if all elements in an iterable are truthy. | ```python<br>all({'*', ' ', ''})   # → False<br>all([' ', ' ', ' '])   # → True``` |
| **`any()`** | Returns `True` if at least one element in an iterable is truthy. | ```python<br>any((1, 0, 0))   # → True<br>any((0, 0, 0))   # → False``` |
| **`ascii()`** | Returns a printable representation of an object, escaping non‑ASCII characters. | ```python<br>ascii('a')   # → "'a'"<br>ascii('ä')   # → "'\\u00e4'"``` |
| **`chr()`** | Returns the character that represents the specified Unicode code point. | ```python<br>chr(9)   # → '\\t'<br>chr(48)  # → '0

## **15. Display Source Documents**

In [58]:
question = "What does the len() function do?"

retrieved_docs = retriever.invoke(question)

context = format_docs(retrieved_docs)

response = llm.invoke(
    prompt.format(
        context=context,
        question=question
    )
)

print("ANSWER:")
print(response.content)

print("\nSOURCES:")
for doc in retrieved_docs:
    print("Page:", doc.metadata.get("page", "Unknown"))

ANSWER:
I could not find the answer in the provided document.

SOURCES:
Page: 12
Page: 11
Page: 1


## **Create a simple chatbot loop**

In [59]:
while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Chatbot stopped.")
        break

    retrieved_docs = retriever.invoke(question)

    context = format_docs(retrieved_docs)

    response = llm.invoke(
        prompt.format(
            context=context,
            question=question
        )
    )

    print("\n🤖 Answer:")
    print(response.content)

    print("\n📄 Sources:")
    for doc in retrieved_docs:
        print(f"Page {doc.metadata.get('page', 'Unknown')}")


Ask a question (type 'exit' to stop): What is Agentic AI?

🤖 Answer:
I could not find the answer in the provided document.

📄 Sources:
Page 4
Page 7
Page 10

Ask a question (type 'exit' to stop): What is the difference between all() and any()?

🤖 Answer:
all() returns True only if every element in the iterable is truthy; any() returns True if at least one element in the iterable is truthy.

📄 Sources:
Page 1
Page 12
Page 11

Ask a question (type 'exit' to stop): What does enumerate() do?

🤖 Answer:
enumerate() is a Python built‑in function that returns an enumerate object, effectively adding a counter to an iterable.

📄 Sources:
Page 7
Page 12
Page 10

Ask a question (type 'exit' to stop): What does bin() do?

🤖 Answer:
bin() converts an integer to a binary string.

📄 Sources:
Page 2
Page 4
Page 12

Ask a question (type 'exit' to stop): exit
Chatbot stopped.


## **16. Conclusion**

### This project demonstrates how RAG can connect LLMs with external knowledge sources. Instead of relying only on the model's pre-trained knowledge, the chatbot retrieves relevant information from the PDF and uses it to generate responses. Through this project, I learned the complete RAG pipeline, including document loading, information extraction, text chunking, embeddings, vector storage, retrieval, prompt construction, and LLM-based answer generation. The same architecture can be extended to multiple PDFs, research papers, company documents, technical documentation, and other domain-specific knowledge bases.

# **Step 1 — Install Gradio**

In [60]:
!pip install -q gradio

## **Step 2 — Your existing RAG pipeline stays**

In [ ]:
documents
chunks
embeddings
vectorstore
retriever
llm
prompt
format_docs

## **Step 3 — Create one RAG function**

In [61]:
def answer_question(question):

    retrieved_docs = retriever.invoke(question)

    context = format_docs(retrieved_docs)

    response = llm.invoke(
        prompt.format(
            context=context,
            question=question
        )
    )

    sources = []

    for doc in retrieved_docs:
        page = doc.metadata.get("page", "Unknown")
        sources.append(f"Page {page}")

    sources = sorted(set(sources))

    return response.content, sources

## **Step 4 — Build the professional Gradio UI**

In [64]:
import os
import uuid
import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate


# =========================================================
# GLOBAL VARIABLES
# =========================================================

current_retriever = None
current_documents = []
current_chunks = []


# =========================================================
# RAG PROMPT
# =========================================================

prompt = ChatPromptTemplate.from_template("""
You are a helpful PDF assistant.

Answer the question ONLY using the context provided below.

If the answer is not available in the PDF, say:
"I could not find the answer in the uploaded PDF."

Context:
{context}

Question:
{question}

Answer:
""")


# =========================================================
# FORMAT DOCUMENTS
# =========================================================

def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )


# =========================================================
# PROCESS PDF
# =========================================================

def process_pdf(pdf_path):

    global current_retriever
    global current_documents
    global current_chunks

    if pdf_path is None:
        return (
            "⚠️ **Please upload a PDF first.**",
            "No document loaded."
        )

    try:

        # ---------------------------------------------
        # 1. Load PDF
        # ---------------------------------------------

        loader = PyPDFLoader(pdf_path)
        documents = loader.load()

        if not documents:
            return (
                "❌ **Could not extract text from this PDF.**",
                "No pages found."
            )

        # ---------------------------------------------
        # 2. Text Splitting
        # ---------------------------------------------

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

        chunks = text_splitter.split_documents(documents)

        # ---------------------------------------------
        # 3. Embeddings
        # ---------------------------------------------

        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        # ---------------------------------------------
        # 4. Create Chroma Vector Database
        # ---------------------------------------------

        collection_name = f"pdf_rag_{uuid.uuid4().hex}"

        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=collection_name
        )

        # ---------------------------------------------
        # 5. Create Retriever
        # ---------------------------------------------

        current_retriever = vectorstore.as_retriever(
            search_kwargs={"k": 3}
        )

        current_documents = documents
        current_chunks = chunks

        # ---------------------------------------------
        # 6. PDF Information
        # ---------------------------------------------

        filename = os.path.basename(pdf_path)
        pages = len(documents)
        total_chunks = len(chunks)

        status = f"""
## ✅ PDF Upload Successful!

**📄 File:** `{filename}`

**📑 Pages:** `{pages}`

**🧩 Chunks:** `{total_chunks}`

**🟢 RAG Status:** Ready

You can now ask questions about this PDF.
"""

        info = f"""
### 📊 Document Information

- **File:** {filename}
- **Pages:** {pages}
- **Chunks:** {total_chunks}
- **Embedding:** `all-MiniLM-L6-v2`
- **Vector Database:** `Chroma`
- **Retriever:** Top 3 chunks
"""

        return status, info

    except Exception as e:

        return (
            f"❌ **Error while processing PDF:**\n\n`{str(e)}`",
            "PDF processing failed."
        )


# =========================================================
# ASK QUESTION
# =========================================================

def ask_question(question):

    global current_retriever

    if current_retriever is None:
        return (
            "⚠️ **Please upload and process a PDF first.**",
            ""
        )

    if question is None or question.strip() == "":
        return (
            "⚠️ **Please enter a question.**",
            ""
        )

    try:

        # ---------------------------------------------
        # Retrieve relevant chunks
        # ---------------------------------------------

        retrieved_docs = current_retriever.invoke(question)

        # ---------------------------------------------
        # Create context
        # ---------------------------------------------

        context = format_docs(retrieved_docs)

        # ---------------------------------------------
        # Ask LLM
        # ---------------------------------------------

        response = llm.invoke(
            prompt.format(
                context=context,
                question=question
            )
        )

        # ---------------------------------------------
        # Sources
        # ---------------------------------------------

        sources = []

        for doc in retrieved_docs:

            page = doc.metadata.get("page", None)

            if page is not None:

                # PyPDFLoader page numbers start from 0
                page_number = page + 1

                sources.append(
                    f"- 📄 Page **{page_number}**"
                )

        sources = list(dict.fromkeys(sources))

        if not sources:
            sources_text = "No source pages found."
        else:
            sources_text = "\n".join(sources)

        return response.content, sources_text

    except Exception as e:

        return (
            f"❌ **Error:** `{str(e)}`",
            ""
        )


# =========================================================
# CLEAR FUNCTION
# =========================================================

def clear_all():

    global current_retriever
    global current_documents
    global current_chunks

    current_retriever = None
    current_documents = []
    current_chunks = []

    return (
        None,
        "### 📄 Document\n\nNo PDF loaded.",
        "### 📊 Document Information\n\nWaiting for PDF...",
        "",
        "",
        ""
    )


# =========================================================
# CUSTOM CSS
# =========================================================

custom_css = """

/* Main container */

.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
}


/* Header */

#main-title {
    text-align: center;
    margin-bottom: 5px;
}

#subtitle {
    text-align: center;
    opacity: 0.75;
}


/* Cards */

.card {
    border-radius: 16px !important;
    padding: 20px !important;
    border: 1px solid #e5e7eb !important;
}


/* Upload area */

.upload-card {
    border-radius: 16px !important;
    padding: 20px !important;
}


/* Footer */

#footer {
    text-align: center;
    margin-top: 30px;
    padding: 20px;
    opacity: 0.75;
    font-size: 14px;
}


/* Status */

#status {
    border-radius: 12px;
    padding: 15px;
}

"""


# =========================================================
# GRADIO APP
# =========================================================

with gr.Blocks(
    title="PDF RAG Assistant",
    css=custom_css
) as demo:

    # -----------------------------------------------------
    # HEADER
    # -----------------------------------------------------

    gr.Markdown(
        "# 📚 PDF RAG Assistant",
        elem_id="main-title"
    )

    gr.Markdown(
        "Upload any PDF and ask questions using Retrieval-Augmented Generation",
        elem_id="subtitle"
    )

    gr.Markdown("---")


    # -----------------------------------------------------
    # MAIN LAYOUT
    # -----------------------------------------------------

    with gr.Row():

        # =================================================
        # LEFT SIDEBAR
        # =================================================

        with gr.Column(
            scale=1,
            elem_classes="card"
        ):

            gr.Markdown("## 📄 Document")

            pdf_file = gr.File(
                label="Upload your PDF",
                file_types=[".pdf"],
                file_count="single",
                type="filepath"
            )

            process_button = gr.Button(
                "⚡ Process PDF",
                variant="primary"
            )

            status = gr.Markdown(
                "### 📄 PDF Status\n\nNo PDF uploaded.",
                elem_id="status"
            )

            gr.Markdown("---")

            document_info = gr.Markdown(
                """
### 📊 Document Information

Waiting for PDF...
"""
            )

            clear_button = gr.Button(
                "🗑️ Clear Document"
            )


        # =================================================
        # RIGHT SIDE
        # =================================================

        with gr.Column(
            scale=2,
            elem_classes="card"
        ):

            gr.Markdown("## 💬 Ask Your PDF")

            question = gr.Textbox(
                label="Your Question",
                placeholder="Example: What is the abs() function?",
                lines=2
            )

            with gr.Row():

                ask_button = gr.Button(
                    "🚀 Ask Question",
                    variant="primary"
                )

                clear_question = gr.Button(
                    "Clear Question"
                )

            gr.Markdown("---")

            gr.Markdown("### 🤖 Answer")

            answer = gr.Markdown(
                "Your answer will appear here..."
            )

            gr.Markdown("### 📚 Sources")

            sources = gr.Markdown(
                "Sources will appear here..."
            )


    # =====================================================
    # EXAMPLE QUESTIONS
    # =====================================================

    gr.Markdown("---")

    gr.Markdown("## 💡 Example Questions")

    gr.Examples(
        examples=[
            ["What is the abs() function?"],
            ["What does the all() function do?"],
            ["What is the difference between all() and any()?"],
            ["What does the bin() function do?"],
            ["What is the purpose of bool()?"],
            ["What does enumerate() do?"]
        ],
        inputs=question
    )


    # =====================================================
    # FOOTER
    # =====================================================

    gr.Markdown(
        """
---
<div id="footer">

### 🤖 PDF RAG Assistant

**Built with:** LangChain • Hugging Face • ChromaDB • Groq • Gradio

**Created by Ajim Mujawar**

</div>
"""
    )


    # =====================================================
    # EVENTS
    # =====================================================

    # Upload PDF
    pdf_file.upload(
        fn=process_pdf,
        inputs=pdf_file,
        outputs=[status, document_info]
    )


    # Process button
    process_button.click(
        fn=process_pdf,
        inputs=pdf_file,
        outputs=[status, document_info]
    )


    # Ask question
    ask_button.click(
        fn=ask_question,
        inputs=question,
        outputs=[answer, sources]
    )


    # Press Enter in question box
    question.submit(
        fn=ask_question,
        inputs=question,
        outputs=[answer, sources]
    )


    # Clear question
    clear_question.click(
        fn=lambda: "",
        outputs=question
    )


    # Clear everything
    clear_button.click(
        fn=clear_all,
        outputs=[
            pdf_file,
            status,
            document_info,
            question,
            answer,
            sources
        ]
    )


# =========================================================
# LAUNCH
# =========================================================

demo.launch(
    share=True
)

/tmp/ipykernel_784/1799370049.py:343: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://16980aadff766bd60a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
